# Assignment 2, Hint 4: strong augmentation

CSCI 6379 · companion to the [Assignment 2 hints page](https://dongchul.kim/csci6379-fall2026/assignment/2/hints).

The hidden images come from **other hospitals**: other endoscopes, other lighting and colours, polyps photographed at other sizes. A network trained on one hospital's images learns that hospital's look along with the polyps. Changing every training batch at random, in the ways another hospital's images would differ, teaches it to rely on the polyp's shape and outline instead. This `augment` flips and turns each image, rotates it by up to 15 degrees, zooms between 0.5x and 1.5x, shifts it, and then changes brightness, contrast, saturation, tint and gamma. The wide zoom matters: in our runs, 0.75x to 1.25x scored 0.641 and 0.5x to 1.5x scored 0.661.

**Where it goes.** `recipe.py` in your notebook: replace the whole `augment` function. The training loop already calls it on every training batch, and never on validation images.

This notebook holds **only the code of this hint**. It is not a training program and it
does not touch the assignment data. Copy the marked cell into your own Assignment 2
notebook, change `RUN_NAME` there, and train as before. The last cell is a small check
you can run here to see the hint working.

## Hint 4: copy this into `recipe.py`, replacing `augment`

In [ ]:
# ---- copy from here into recipe.py: it replaces the whole augment function ----
import math
import torch
import torch.nn.functional as F


def augment(x, m):
    # x: images (B, 3, 256, 256), floats in [0, 1].  m: masks (B, 1, 256, 256), 0.0 or 1.0.
    # Called on every TRAINING batch, on the GPU. Each image in the batch gets its own random change,
    # and a new one every epoch, so the network never sees exactly the same picture twice.
    B, dev = len(x), x.device

    # ---------- geometric changes: applied to the image AND its mask in exactly the same way ----------
    # (if the image turns and the mask does not, the mask no longer marks the polyp)
    f = torch.rand(B, device=dev) < 0.5                  # pick about half the batch ...
    x[f], m[f] = x[f].flip(-1), m[f].flip(-1)            # ... and mirror them left to right
    f = torch.rand(B, device=dev) < 0.5
    x[f], m[f] = x[f].flip(-2), m[f].flip(-2)            # mirror about half of them top to bottom
    k = int(torch.randint(0, 4, (1,)))                   # turn the whole batch by 0, 90, 180 or 270 degrees
    x, m = torch.rot90(x, k, (2, 3)), torch.rot90(m, k, (2, 3))

    # One random affine transform per image: rotation, zoom and shift together.
    ang = (torch.rand(B, device=dev) - 0.5) * (math.pi / 6)                    # rotation: -15 to +15 degrees
    sc = torch.exp(math.log(0.5) + math.log(3.0) * torch.rand(B, device=dev))  # zoom: 0.5x to 1.5x, drawn on a
                                                                               # log scale so shrinking and
                                                                               # enlarging are equally likely
    tx, ty = (torch.rand(2, B, device=dev) - 0.5) * 0.3                        # shift: up to 7.5% of the width
                                                                               # in each direction
    # affine_grid wants, for every OUTPUT pixel, where to read in the INPUT. Dividing by the zoom
    # factor reads from a smaller area (zoom in) when sc > 1 and a larger one (zoom out) when sc < 1.
    th = torch.stack([torch.stack([torch.cos(ang) / sc, -torch.sin(ang) / sc, tx], 1),
                      torch.stack([torch.sin(ang) / sc,  torch.cos(ang) / sc, ty], 1)], 1)   # (B, 2, 3)
    g = F.affine_grid(th, list(x.shape), align_corners=False)
    # bilinear for the image (smooth), nearest for the mask (it must stay 0 or 1);
    # anything that comes from outside the picture is filled with 0 (black, and "not polyp")
    x = F.grid_sample(x, g, mode="bilinear", padding_mode="zeros", align_corners=False)
    m = (F.grid_sample(m, g, mode="nearest", padding_mode="zeros", align_corners=False) > 0.5).float()

    # ---------- colour changes: the image ONLY (the mask has no colour) ----------
    # Each imitates what another endoscope, light source or image pipeline does to the picture.
    r = lambda *s: torch.rand(*s, device=dev) * 2 - 1    # random numbers, uniform in [-1, 1]
    x = x * (1 + 0.4 * r(B, 1, 1, 1))                    # brightness: everything 0.6x to 1.4x as bright
    mu = x.mean((1, 2, 3), keepdim=True)
    x = (x - mu) * (1 + 0.4 * r(B, 1, 1, 1)) + mu        # contrast: stretch or squeeze around the image's mean
    gray = x.mean(1, keepdim=True)
    x = gray + (x - gray) * (1 + 0.4 * r(B, 1, 1, 1))    # saturation: towards grey (duller) or away (more vivid)
    x = x * (1 + 0.15 * r(B, 3, 1, 1))                   # tint: a different gain for R, G and B (white balance)
    x = x.clamp(0, 1) ** torch.exp(0.4 * r(B, 1, 1, 1))  # gamma: brighten or darken the mid-tones
    return x.clamp(0, 1), m                              # back into [0, 1], as the loop expects

# ---- copy up to here ----

## Check (run here)

A made-up picture (a colour gradient with a round "polyp") stands in for a training image. The cell runs `augment` five times on it and draws the mask's outline in green: the outline must turn, flip and zoom with the picture, and the colour changes must not move it.

In [ ]:
import matplotlib.pyplot as plt
H = W = 256
yy, xx = torch.meshgrid(torch.linspace(0, 1, H), torch.linspace(0, 1, W), indexing="ij")
img = torch.stack([0.8 - 0.3 * yy, 0.35 + 0.2 * xx, 0.3 + 0.1 * yy])          # a pinkish gradient
mask = (((xx - 0.65) ** 2 + (yy - 0.35) ** 2) < 0.12 ** 2).float()               # a round "polyp"
img = torch.where(mask.bool(), torch.tensor([0.9, 0.45, 0.4])[:, None, None], img)
x = img[None].repeat(5, 1, 1, 1).clone(); m = mask[None, None].repeat(5, 1, 1, 1).clone()
torch.manual_seed(1)
xa, ma = augment(x, m)

fig, ax = plt.subplots(1, 6, figsize=(15, 2.8))
for a, (pic, msk, t) in zip(ax, [(img, mask, "original")] + [(xa[i], ma[i, 0], f"augmented {i + 1}") for i in range(5)]):
    a.imshow(pic.permute(1, 2, 0)); a.contour(msk, [0.5], colors="lime"); a.set_title(t); a.axis("off")
plt.show()
assert xa.shape == x.shape and ma.shape == m.shape and set(ma.unique().tolist()) <= {0.0, 1.0}
print("OK: shapes kept, the mask is still 0/1, and the outline moves with the picture.")